In [2]:
from sqlitesearch import TextSearchIndex

sqlite_index = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="faq.db"
)

In [3]:
sqlite_index.count()

83

In [4]:
results = sqlite_index.search("Can I still join the course after it started?", num_results=5)
[doc["question"] for doc in results]

['I just discovered the course. Can I still join?',
 'I missed the first homework - can I still get a certificate?',
 'How do I start using Google Gemini models in the Module 1 notebook through the OpenAI-compatible endpoint?',
 'Homework: Why does the content keep changing?',
 'What happens to code saved in Codespaces if I do not commit it?']

In [8]:
from rag_helper import RAGBase
from openai import OpenAI
import os
from dotenv import load_dotenv
load_dotenv()
openai_client = OpenAI(    base_url="https://openrouter.ai/api/v1",
    # Automatically pulls the key from your .env file
    api_key=os.getenv("OPENROUTER_API_KEY") )
assistant = RAGBase(
    index=sqlite_index,
    llm_client=openai_client,
)

In [9]:
answer = assistant.rag("Can I still join the course after it started?")
print(answer)

Yes—you can still join the course after it has started. Just keep in mind that to receive a certificate you’ll need to submit your project while the submission period is still open.


```mermaid
flowchart TD

    subgraph RAG["RAG ASSISTANT"]
        U([🙂 User])
        APP[Application]
        DOCS[[D1 ... D5]]
        PROMPT[Build Prompt<br/>Question + Context]
        LLM[LLM]
        ANSWER([Answer])

        U -->|Question| APP
        DOCS --> APP
        APP --> PROMPT
        PROMPT --> LLM
        LLM --> ANSWER
        ANSWER --> U
    end

    subgraph KB["KNOWLEDGE BASE"]
        DB[(DB)]
    end

    APP -->|Query| DB
    DB -->|Retrieved Data| DOCS
```